# Mostrar cartas:

Función que muestra una carta por pantalla, se usa para debug

In [ ]:
from IPython.display import display, HTML

def display_card(url):
    # Ahora que tenemos la URL real, generamos el HTML
    html_code = f"""
    <div style="width: 300px; border-radius: 15px; overflow: hidden; box-shadow: 0 8px 16px rgba(0,0,0,0.3);">
        <img src="{url}" alt="Carta" style="width:100%; display: block;">
    </div>
    """
    display(HTML(html_code))

## Crear embeddings:

funcionalidad para crear embeddings apartir del texto y el nombre de una carta

In [ ]:
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


df = pd.read_csv("cards_final_with_xp.csv")

batch_size=32
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

try:
    texts = [f"{name} \n {text}" for name, text in zip(df["name"], df["text"])]
    query_embeddings = model.encode(texts, convert_to_numpy=True, batch_size=batch_size).astype(np.float32)
finally:
    torch.cuda.empty_cache()

In [ ]:
query_embeddings.shape

# Embeddings a Redis
Función que recibe un embedding como un array de numpy y devuelve un blob que se le puede pasar a redis

In [ ]:
import numpy as np

def to_blob(embedding: np.array) -> bytes:
    """
    Converts embedding to blob.
    :param embedding: embedding
    :return: blob
    """
    return embedding.astype(np.float32).tobytes()

# Iniciar conexión con redis

In [ ]:
import redis

r = redis.Redis(host='redis', port=6379)

# **Objetivo I**

### **Tarea 1**. Buscar la estructura de datos más apropiada para la cache.

Para esta tarea, la estructura de datos más apropiada para la cache en Redis sería una **tabla Hash**. Esto es ideal para almacenar la información de cada carta, ya que cada carta puede tener múltiples atributos (nombre, texto, tipo, etc.) que pueden ser almacenados como campos dentro del Hash.  Además, los Hashes permiten un acceso rápido a los datos, lo que es crucial para una cache eficiente.


### **Tarea 2**. Escribir una función Python que reciba un fichero .csv con un conjunto de ejemplo de cartas y las cargue en una base de datos Redis usando la estructura de datos seleccionada en el paso anterior.


In [ ]:
df = pd.read_csv("cards_final_with_xp.csv")

In [ ]:
df.head()

In [ ]:
def cargar_cartas_en_redis(df):
    
    for i in range(len(df)):
        
        code = df.loc[i, "code"]
        name = df.loc[i, "name"]
        text = df.loc[i, "text"]
        type_code = df.loc[i, "type_code"]
        traits = df.loc[i, "traits"]
        pack_code = df.loc[i, "pack_code"]
        faction_code = df.loc[i, "faction_code"]
        xp = df.loc[i, "xp"]
        illustrator = df.loc[i, "illustrator"]
        image_url = df.loc[i, "image_url"]
        
        r.hset('carta:' + code, mapping={
            "name": name,
            "text": text,
            "type_code": type_code,
            "traits": traits,
            "pack_code": pack_code,
            "faction_code": faction_code,
            "xp": xp,
            "illustrator": illustrator,
            "image_url": image_url
        })

In [ ]:
def cargar_cartas_en_redis(df: pd.DataFrame) -> None:
    """
    Carga las cartas de un DataFrame en Redis.

    Args:
        df (pd.DataFrame): DataFrame que contiene las cartas.
    """
    for _, serie in df.iterrows():

        data = serie.to_dict()
        code = data.pop("code", None)
        
        if code:
            key = f"card:{code}"
            
        r.hset(key, mapping=data)

### **Tarea 3**. Escribir funciones python que permita realizar cada una de las acciones. Implementar una por acción.

#### 1. Saber si una carta está en la cache por su campo code.

In [ ]:
def carta_en_cache(code: str) -> bool:
    """
    Verifica si una carta está en la cache por su campo code.

    Args:
        code (str): El código de la carta a verificar.

    Returns:
        bool: True si la carta está en la cache, False en caso contrario.
    """
    key = f"card:{code}"
    return r.exists(key) == 1

# **Objetivo II** 

# **Objetivo III**